<a href="https://colab.research.google.com/github/CodeHunterOfficial/ABCD_ASPNETCORE/blob/main/TajikNLP/RUNG_DataSet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
FULL_HF_TOKEN=userdata.get('FULL_HF_TOKEN')

In [ ]:
FULL_HF_TOKEN

#Полный объединённый скрипт для татарского датасета

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Скрипт для Google Colab:
- Загружает gated датасет TatarNLPWorld/tatar-english-russian-corpus
- Создаёт два датасета в формате JSON:
  1. Русский → Татарский (ru_to_tt.json)
  2. Английский → Татарский (en_to_tt.json)
- Автоматически создаёт подвыборки: 200, 500, 1000 записей
- Формат:
  {
    "sources": ["предложение 1", "предложение 2", ...],
    "references": ["перевод 1", "перевод 2", ...],
    "source_lang": "ru",
    "target_lang": "tt"
  }
"""

import json
import random
from pathlib import Path

# Установка библиотеки datasets (если ещё не установлена)
!pip install datasets -q

from google.colab import userdata
from datasets import load_dataset

# ========== 1. ПОДКЛЮЧЕНИЕ ТОКЕНА ==========
print("=" * 60)
print("ШАГ 1: ПОДКЛЮЧЕНИЕ ТОКЕНА")
print("=" * 60)

try:
    FULL_HF_TOKEN = userdata.get('FULL_HF_TOKEN')
    print("✅ Токен успешно загружен из userdata")
except Exception as e:
    print(f"❌ Ошибка загрузки токена: {e}")
    FULL_HF_TOKEN = input("Введите ваш HF токен: ").strip()

# ========== 2. КОНФИГУРАЦИЯ ==========
print("\n" + "=" * 60)
print("ШАГ 2: КОНФИГУРАЦИЯ")
print("=" * 60)

DATASET_NAME = "TatarNLPWorld/tatar-english-russian-corpus"
SPLIT = "full"  # или "sample"
OUTPUT_DIR = Path("./bilingual_datasets")
FILTER_EMPTY_RUSSIAN = True  # True - удалить пустые русские строки
SAMPLE_SIZES = [200, 500, 1000]  # Размеры подвыборок

print(f"📌 Датасет: {DATASET_NAME}")
print(f"📌 Сплит: {SPLIT}")
print(f"📌 Выходная папка: {OUTPUT_DIR}")
print(f"📌 Фильтровать пустые русские строки: {FILTER_EMPTY_RUSSIAN}")
print(f"📌 Размеры подвыборок: {SAMPLE_SIZES}")

# ========== 3. ЗАГРУЗКА ДАТАСЕТА ==========
print("\n" + "=" * 60)
print("ШАГ 3: ЗАГРУЗКА ДАТАСЕТА")
print("=" * 60)

print(f"📥 Загружаем датасет {DATASET_NAME} (split={SPLIT})...")
try:
    ds = load_dataset(DATASET_NAME, split=SPLIT, token=FULL_HF_TOKEN)
    print(f"✅ Датасет загружен! Количество записей: {len(ds):,}")
    print(f"📊 Колонки: {ds.column_names}")
except Exception as e:
    print(f"❌ Ошибка загрузки: {e}")
    raise

# Фильтрация пустых русских строк (опционально)
if FILTER_EMPTY_RUSSIAN:
    before = len(ds)
    ds = ds.filter(lambda x: x["russian_text"] != "")
    after = len(ds)
    print(f"🧹 Удалено записей с пустым russian_text: {before - after}")
    print(f"✅ После фильтрации: {after:,} записей")

# ========== 4. ПРЕОБРАЗОВАНИЕ В СПИСКИ PYTHON ==========
print("\n" + "=" * 60)
print("ШАГ 4: ПРЕОБРАЗОВАНИЕ В СПИСКИ PYTHON")
print("=" * 60)

print("🔄 Преобразуем колонки в списки...")
english_texts = list(ds["english_text"])
tatar_texts = list(ds["tatar_text"])
russian_texts = list(ds["russian_text"])
print(f"✅ Преобразовано {len(english_texts):,} записей")

# ========== 5. СОЗДАНИЕ JSON-ФАЙЛОВ ==========
print("\n" + "=" * 60)
print("ШАГ 5: СОЗДАНИЕ JSON-ФАЙЛОВ")
print("=" * 60)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 5.1. Русско-татарский датасет (ru → tt)
print("🔄 Создаём русско-татарский датасет (ru → tt)...")
ru_to_tt = {
    "sources": russian_texts,
    "references": tatar_texts,
    "source_lang": "ru",
    "target_lang": "tt"
}
print(f"✅ Создано {len(ru_to_tt['sources']):,} записей")

# 5.2. Англо-татарский датасет (en → tt)
print("🔄 Создаём англо-татарский датасет (en → tt)...")
en_to_tt = {
    "sources": english_texts,
    "references": tatar_texts,
    "source_lang": "en",
    "target_lang": "tt"
}
print(f"✅ Создано {len(en_to_tt['sources']):,} записей")

# ========== 6. СОХРАНЕНИЕ ПОЛНЫХ JSON-ФАЙЛОВ ==========
print("\n" + "=" * 60)
print("ШАГ 6: СОХРАНЕНИЕ ПОЛНЫХ JSON-ФАЙЛОВ")
print("=" * 60)

print(f"💾 Сохраняем файлы в {OUTPUT_DIR}/")

# Сохраняем русско-татарский
ru_tt_file = OUTPUT_DIR / "ru_to_tt.json"
print(f"   📝 {ru_tt_file.name}...")
with open(ru_tt_file, "w", encoding="utf-8") as f:
    json.dump(ru_to_tt, f, ensure_ascii=False, indent=2)

# Сохраняем англо-татарский
en_tt_file = OUTPUT_DIR / "en_to_tt.json"
print(f"   📝 {en_tt_file.name}...")
with open(en_tt_file, "w", encoding="utf-8") as f:
    json.dump(en_to_tt, f, ensure_ascii=False, indent=2)

print("✅ Полные датасеты сохранены!")

# ========== 7. СОЗДАНИЕ ПОДВЫБОРОК ==========
print("\n" + "=" * 60)
print("ШАГ 7: СОЗДАНИЕ ПОДВЫБОРОК")
print("=" * 60)

def extract_sample(data, n, random_seed=42):
    """
    Извлекает случайную подвыборку из n записей
    """
    total = len(data['sources'])

    if n >= total:
        print(f"⚠️ Запрошено {n} записей, но доступно только {total}. Берём все.")
        n = total

    # Создаём индексы и перемешиваем
    indices = list(range(total))
    random.seed(random_seed)
    random.shuffle(indices)
    selected_indices = indices[:n]

    # Создаём новую выборку
    sample = {
        "sources": [data['sources'][i] for i in selected_indices],
        "references": [data['references'][i] for i in selected_indices],
        "source_lang": data['source_lang'],
        "target_lang": data['target_lang']
    }

    return sample

# Создаём папку для подвыборок
samples_dir = OUTPUT_DIR / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)

print(f"📊 Создаём подвыборки в папке: {samples_dir}")
print("-" * 40)

sample_files = {}

for size in SAMPLE_SIZES:
    print(f"\n📊 Создаём подвыборку из {size} записей...")

    # Русско-татарский
    ru_sample = extract_sample(ru_to_tt, size)
    ru_sample_file = samples_dir / f"ru_to_tt_{size}.json"
    with open(ru_sample_file, "w", encoding="utf-8") as f:
        json.dump(ru_sample, f, ensure_ascii=False, indent=2)
    print(f"   ✅ Русско-татарский: {len(ru_sample['sources'])} записей -> {ru_sample_file.name}")

    # Англо-татарский
    en_sample = extract_sample(en_to_tt, size)
    en_sample_file = samples_dir / f"en_to_tt_{size}.json"
    with open(en_sample_file, "w", encoding="utf-8") as f:
        json.dump(en_sample, f, ensure_ascii=False, indent=2)
    print(f"   ✅ Англо-татарский: {len(en_sample['sources'])} записей -> {en_sample_file.name}")

    sample_files[str(size)] = {
        "ru": {
            "file": ru_sample_file.name,
            "num_examples": len(ru_sample['sources'])
        },
        "en": {
            "file": en_sample_file.name,
            "num_examples": len(en_sample['sources'])
        }
    }

print("\n✅ Все подвыборки созданы!")

# ========== 8. СОХРАНЕНИЕ МЕТАДАННЫХ ==========
print("\n" + "=" * 60)
print("ШАГ 8: СОХРАНЕНИЕ МЕТАДАННЫХ")
print("=" * 60)

metadata = {
    "full_datasets": {
        "ru_to_tt": {
            "source_lang": "ru",
            "target_lang": "tt",
            "num_examples": len(ru_to_tt["sources"]),
            "file": "ru_to_tt.json"
        },
        "en_to_tt": {
            "source_lang": "en",
            "target_lang": "tt",
            "num_examples": len(en_to_tt["sources"]),
            "file": "en_to_tt.json"
        }
    },
    "samples": sample_files,
    "original_dataset": {
        "name": DATASET_NAME,
        "split": SPLIT,
        "total_records": len(ds),
        "filtered_empty_russian": FILTER_EMPTY_RUSSIAN,
        "column_names": ds.column_names
    },
    "statistics": {
        "empty_russian_removed": before - after if FILTER_EMPTY_RUSSIAN else 0,
        "final_records": len(ds)
    }
}

with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ Метаданные сохранены!")

# ========== 9. ПРОВЕРКА И СТАТИСТИКА ==========
print("\n" + "=" * 60)
print("ШАГ 9: СТАТИСТИКА И ПРОВЕРКА")
print("=" * 60)

print("\n📊 СТАТИСТИКА ДАТАСЕТА:")
print(f"   📁 Всего записей в исходном датасете: {len(ds):,}")
print(f"   📁 Русско-татарский: {len(ru_to_tt['sources']):,} записей")
print(f"   📁 Англо-татарский: {len(en_to_tt['sources']):,} записей")

print("\n📁 СОЗДАННЫЕ ФАЙЛЫ:")
print(f"   📄 Полные датасеты:")
print(f"      - ru_to_tt.json ({len(ru_to_tt['sources']):,} записей)")
print(f"      - en_to_tt.json ({len(en_to_tt['sources']):,} записей)")
print(f"   📁 Подвыборки в папке: {samples_dir}")
for size, info in sample_files.items():
    print(f"      📄 ru_to_tt_{size}.json ({info['ru']['num_examples']} записей)")
    print(f"      📄 en_to_tt_{size}.json ({info['en']['num_examples']} записей)")

# Проверяем первые 3 предложения из каждого датасета
print("\n🔍 ПРИМЕРЫ (первые 3 предложения):")

print("\n--- Русско-татарский (ru_to_tt) ---")
for i in range(min(3, len(ru_to_tt["sources"]))):
    print(f"  {i+1}. Источник: {ru_to_tt['sources'][i][:100]}...")
    print(f"     Перевод: {ru_to_tt['references'][i][:100]}...")

print("\n--- Англо-татарский (en_to_tt) ---")
for i in range(min(3, len(en_to_tt["sources"]))):
    print(f"  {i+1}. Источник: {en_to_tt['sources'][i][:100]}...")
    print(f"     Перевод: {en_to_tt['references'][i][:100]}...")

# Показываем структуру JSON
print("\n📄 СТРУКТУРА JSON:")
print(json.dumps({
    "sources": ["предложение 1", "предложение 2", "..."],
    "references": ["перевод 1", "перевод 2", "..."],
    "source_lang": "ru",
    "target_lang": "tt"
}, ensure_ascii=False, indent=2))

# ========== 10. ИНФОРМАЦИЯ ДЛЯ СКАЧИВАНИЯ ==========
print("\n" + "=" * 60)
print("ШАГ 10: СКАЧИВАНИЕ ФАЙЛОВ")
print("=" * 60)

print(f"\n📁 Файлы сохранены в папке: {OUTPUT_DIR.absolute()}")

print("\n📥 Команды для скачивания полных датасетов:")
print("   from google.colab import files")
print("   files.download('bilingual_datasets/ru_to_tt.json')")
print("   files.download('bilingual_datasets/en_to_tt.json')")

print("\n📥 Команды для скачивания подвыборок:")
for size in SAMPLE_SIZES:
    print(f"   files.download('bilingual_datasets/samples/ru_to_tt_{size}.json')")
    print(f"   files.download('bilingual_datasets/samples/en_to_tt_{size}.json')")

print(f"\n📥 files.download('bilingual_datasets/metadata.json')")

# Автоматическое скачивание
download = input("\n⬇️ Скачать файлы автоматически? (y/n): ").strip().lower()
if download == 'y':
    from google.colab import files

    # Скачиваем полные датасеты
    print("\n📥 Скачиваем полные датасеты...")
    files.download(str(ru_tt_file))
    files.download(str(en_tt_file))

    # Скачиваем подвыборки
    print("\n📥 Скачиваем подвыборки...")
    for size in SAMPLE_SIZES:
        ru_sample_file = samples_dir / f"ru_to_tt_{size}.json"
        en_sample_file = samples_dir / f"en_to_tt_{size}.json"
        if ru_sample_file.exists():
            files.download(str(ru_sample_file))
        if en_sample_file.exists():
            files.download(str(en_sample_file))

    # Скачиваем метаданные
    print("\n📥 Скачиваем metadata.json...")
    files.download(str(OUTPUT_DIR / "metadata.json"))

    print("✅ Все файлы скачаны!")

print("\n" + "=" * 60)
print("🎉 ГОТОВО!")
print("=" * 60)

# ========== 11. ИТОГОВАЯ ИНФОРМАЦИЯ ==========
print("\n" + "=" * 60)
print("📊 ИТОГОВАЯ ИНФОРМАЦИЯ О ДАТАСЕТЕ")
print("=" * 60)

print(f"""
📋 Сводка:
   • Всего записей: {len(ds):,}
   • Русско-татарский: {len(ru_to_tt['sources']):,} записей
   • Англо-татарский: {len(en_to_tt['sources']):,} записей
   • Удалено пустых русских строк: {before - after if FILTER_EMPTY_RUSSIAN else 0}

📁 Сохранённые файлы:
   • Полные датасеты: 2 файла
   • Подвыборки: {len(SAMPLE_SIZES) * 2} файлов
   • Метаданные: metadata.json
""")

# Скрипт для Tajik-Persian датасета

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Скрипт для Google Colab:
- Загружает датасет TajikNLPWorld/TajPersParallelCorpusFull
- Создаёт JSON-датасет в формате:
  {
    "sources": ["предложение 1", "предложение 2", ...],
    "references": ["перевод 1", "перевод 2", ...],
    "source_lang": "tg",
    "target_lang": "fa"
  }
- Автоматически создаёт подвыборки: 200, 500, 1000, 10000 записей
"""

import json
import random
from pathlib import Path

# Установка библиотеки datasets (если ещё не установлена)
!pip install datasets -q

from google.colab import userdata
from datasets import load_dataset

# ========== 1. ПОДКЛЮЧЕНИЕ ТОКЕНА ==========
print("=" * 60)
print("ШАГ 1: ПОДКЛЮЧЕНИЕ ТОКЕНА")
print("=" * 60)

try:
    FULL_HF_TOKEN = userdata.get('FULL_HF_TOKEN')
    print("✅ Токен успешно загружен из userdata")
except Exception as e:
    print(f"❌ Ошибка загрузки токена: {e}")
    FULL_HF_TOKEN = input("Введите ваш HF токен: ").strip()

# ========== 2. КОНФИГУРАЦИЯ ==========
print("\n" + "=" * 60)
print("ШАГ 2: КОНФИГУРАЦИЯ")
print("=" * 60)

DATASET_NAME = "TajikNLPWorld/TajPersParallelCorpusFull"
SPLIT = "train"  # или "full"
OUTPUT_DIR = Path("./tajik_persian_datasets")

# Размеры подвыборок
SAMPLE_SIZES = [200, 500, 1000, 10000]

# Языки
SOURCE_LANG = "tg"  # таджикский
TARGET_LANG = "fa"  # персидский

print(f"📌 Датасет: {DATASET_NAME}")
print(f"📌 Сплит: {SPLIT}")
print(f"📌 Выходная папка: {OUTPUT_DIR}")
print(f"📌 Язык источника: {SOURCE_LANG}")
print(f"📌 Язык перевода: {TARGET_LANG}")
print(f"📌 Размеры подвыборок: {SAMPLE_SIZES}")

# ========== 3. ЗАГРУЗКА ДАТАСЕТА ==========
print("\n" + "=" * 60)
print("ШАГ 3: ЗАГРУЗКА ДАТАСЕТА")
print("=" * 60)

print(f"📥 Загружаем датасет {DATASET_NAME} (split={SPLIT})...")
try:
    ds = load_dataset(DATASET_NAME, split=SPLIT, token=FULL_HF_TOKEN)
    print(f"✅ Датасет загружен! Количество записей: {len(ds):,}")
    print(f"📊 Колонки: {ds.column_names}")
except Exception as e:
    print(f"❌ Ошибка загрузки: {e}")
    raise

# ========== 4. ОПРЕДЕЛЕНИЕ КОЛОНОК ==========
print("\n" + "=" * 60)
print("ШАГ 4: ОПРЕДЕЛЕНИЕ КОЛОНОК")
print("=" * 60)

# Автоматическое определение названий колонок
if 'tajik' in ds.column_names and 'farsi' in ds.column_names:
    SOURCE_COL = 'tajik'
    TARGET_COL = 'farsi'
elif 'tajik_text' in ds.column_names and 'persian_text' in ds.column_names:
    SOURCE_COL = 'tajik_text'
    TARGET_COL = 'persian_text'
elif 'tajik' in ds.column_names and 'persian' in ds.column_names:
    SOURCE_COL = 'tajik'
    TARGET_COL = 'persian'
else:
    # Если колонки называются иначе, берём первые две
    SOURCE_COL = ds.column_names[0]
    TARGET_COL = ds.column_names[1]
    print(f"⚠️ Используем колонки: '{SOURCE_COL}' и '{TARGET_COL}'")

print(f"📌 Колонка источника: {SOURCE_COL}")
print(f"📌 Колонка перевода: {TARGET_COL}")

# ========== 5. ПРЕОБРАЗОВАНИЕ В СПИСКИ PYTHON ==========
print("\n" + "=" * 60)
print("ШАГ 5: ПРЕОБРАЗОВАНИЕ В СПИСКИ PYTHON")
print("=" * 60)

print("🔄 Преобразуем колонки в списки...")
source_texts = list(ds[SOURCE_COL])
target_texts = list(ds[TARGET_COL])

# Проверяем на пустые значения
empty_source = sum(1 for x in source_texts if x == "" or x is None)
empty_target = sum(1 for x in target_texts if x == "" or x is None)

print(f"✅ Преобразовано {len(source_texts):,} записей")
print(f"📊 Пустых записей в источнике: {empty_source}")
print(f"📊 Пустых записей в переводе: {empty_target}")

# ========== 6. СОЗДАНИЕ ПОЛНОГО JSON-ФАЙЛА ==========
print("\n" + "=" * 60)
print("ШАГ 6: СОЗДАНИЕ ПОЛНОГО JSON-ФАЙЛА")
print("=" * 60)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Создаём полный датасет
print("🔄 Создаём полный датасет (таджикский → персидский)...")
full_dataset = {
    "sources": source_texts,
    "references": target_texts,
    "source_lang": SOURCE_LANG,
    "target_lang": TARGET_LANG
}
print(f"✅ Создано {len(full_dataset['sources']):,} записей")

# Сохраняем полный датасет
full_file = OUTPUT_DIR / f"{SOURCE_LANG}_to_{TARGET_LANG}_full.json"
print(f"💾 Сохраняем {full_file.name}...")
with open(full_file, "w", encoding="utf-8") as f:
    json.dump(full_dataset, f, ensure_ascii=False, indent=2)
print(f"✅ Файл сохранён: {full_file.name}")

# ========== 7. СОЗДАНИЕ ПОДВЫБОРОК ==========
print("\n" + "=" * 60)
print("ШАГ 7: СОЗДАНИЕ ПОДВЫБОРОК")
print("=" * 60)

def extract_sample(data, n, random_seed=42):
    """
    Извлекает случайную подвыборку из n записей

    Args:
        data: словарь с ключами 'sources', 'references', 'source_lang', 'target_lang'
        n: количество записей для извлечения
        random_seed: seed для воспроизводимости

    Returns:
        словарь с подвыборкой
    """
    total = len(data['sources'])

    if n >= total:
        print(f"⚠️ Запрошено {n} записей, но доступно только {total}. Берём все.")
        n = total

    # Создаём индексы и перемешиваем
    indices = list(range(total))
    random.seed(random_seed)
    random.shuffle(indices)
    selected_indices = indices[:n]

    # Создаём новую выборку
    sample = {
        "sources": [data['sources'][i] for i in selected_indices],
        "references": [data['references'][i] for i in selected_indices],
        "source_lang": data['source_lang'],
        "target_lang": data['target_lang']
    }

    return sample

# Создаём папку для подвыборок
samples_dir = OUTPUT_DIR / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)

print(f"📊 Создаём подвыборки в папке: {samples_dir}")
print("-" * 40)

sample_files = {}
for size in SAMPLE_SIZES:
    print(f"\n📊 Создаём подвыборку из {size:,} записей...")

    sample = extract_sample(full_dataset, size)
    sample_file = samples_dir / f"{SOURCE_LANG}_to_{TARGET_LANG}_{size}.json"

    with open(sample_file, "w", encoding="utf-8") as f:
        json.dump(sample, f, ensure_ascii=False, indent=2)

    print(f"   ✅ Создано {len(sample['sources']):,} записей -> {sample_file.name}")
    sample_files[str(size)] = {
        "file": sample_file.name,
        "num_examples": len(sample['sources'])
    }

# ========== 8. СОХРАНЕНИЕ МЕТАДАННЫХ ==========
print("\n" + "=" * 60)
print("ШАГ 8: СОХРАНЕНИЕ МЕТАДАННЫХ")
print("=" * 60)

metadata = {
    "full_dataset": {
        "source_lang": SOURCE_LANG,
        "target_lang": TARGET_LANG,
        "num_examples": len(full_dataset["sources"]),
        "file": full_file.name,
        "source_col": SOURCE_COL,
        "target_col": TARGET_COL,
        "empty_source": empty_source,
        "empty_target": empty_target
    },
    "samples": sample_files,
    "original_dataset": {
        "name": DATASET_NAME,
        "split": SPLIT,
        "total_records": len(ds),
        "column_names": ds.column_names
    }
}

# Добавляем статистику по категориям (если есть)
if 'category' in ds.column_names:
    from collections import Counter
    categories = list(ds['category'])
    category_counts = Counter(categories)
    metadata["category_stats"] = {
        cat: count for cat, count in category_counts.most_common(20)
    }

with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ Метаданные сохранены!")

# ========== 9. ПРОВЕРКА И СТАТИСТИКА ==========
print("\n" + "=" * 60)
print("ШАГ 9: СТАТИСТИКА И ПРОВЕРКА")
print("=" * 60)

print("\n📊 СТАТИСТИКА ДАТАСЕТА:")
print(f"   📁 Всего записей: {len(ds):,}")
print(f"   📁 Таджикско-персидский: {len(full_dataset['sources']):,} записей")
print(f"   📁 Пустых записей: {empty_source + empty_target}")

print("\n📁 СОЗДАННЫЕ ФАЙЛЫ:")
print(f"   📄 Полный датасет: {full_file.name} ({len(full_dataset['sources']):,} записей)")
print(f"   📁 Подвыборки в папке: {samples_dir}")
for size, info in sample_files.items():
    print(f"      📄 {info['file']} ({info['num_examples']:,} записей)")

# Проверяем первые 3 предложения
print("\n🔍 ПРИМЕРЫ (первые 3 предложения):")
print("\n--- Таджикский → Персидский ---")
for i in range(min(3, len(full_dataset["sources"]))):
    print(f"  {i+1}. Источник (tg): {full_dataset['sources'][i][:100]}...")
    print(f"     Перевод (fa): {full_dataset['references'][i][:100]}...")
    print()

# Показываем структуру JSON
print("📄 СТРУКТУРА JSON:")
print(json.dumps({
    "sources": ["предложение 1", "предложение 2", "..."],
    "references": ["перевод 1", "перевод 2", "..."],
    "source_lang": SOURCE_LANG,
    "target_lang": TARGET_LANG
}, ensure_ascii=False, indent=2))

# Выводим топ-категории (если есть)
if 'category' in ds.column_names:
    print("\n📊 ТОП-10 КАТЕГОРИЙ:")
    for cat, count in category_counts.most_common(10):
        percentage = (count / len(categories)) * 100
        print(f"   {cat}: {count:,} ({percentage:.2f}%)")

# ========== 10. СКАЧИВАНИЕ ФАЙЛОВ ==========
print("\n" + "=" * 60)
print("ШАГ 10: СКАЧИВАНИЕ ФАЙЛОВ")
print("=" * 60)

print(f"\n📁 Файлы сохранены в папке: {OUTPUT_DIR.absolute()}")

print("\n📥 Команды для скачивания:")
print("   from google.colab import files")
print(f"   files.download('{OUTPUT_DIR}/{full_file.name}')")
print("\n   # Подвыборки:")
for size, info in sample_files.items():
    print(f"   files.download('{OUTPUT_DIR}/samples/{info['file']}')")
print(f"   files.download('{OUTPUT_DIR}/metadata.json')")

# Автоматическое скачивание
download = input("\n⬇️ Скачать файлы автоматически? (y/n): ").strip().lower()
if download == 'y':
    from google.colab import files

    # Скачиваем полный датасет
    print(f"\n📥 Скачиваем {full_file.name}...")
    files.download(str(full_file))

    # Скачиваем подвыборки
    print("\n📥 Скачиваем подвыборки...")
    for size, info in sample_files.items():
        sample_file = samples_dir / info['file']
        if sample_file.exists():
            print(f"   Скачиваем {info['file']}...")
            files.download(str(sample_file))

    # Скачиваем метаданные
    print("\n📥 Скачиваем metadata.json...")
    files.download(str(OUTPUT_DIR / "metadata.json"))

    print("✅ Все файлы скачаны!")

print("\n" + "=" * 60)
print("🎉 ГОТОВО!")
print("=" * 60)

# Полный скрипт для Arabic-Russian датасета (с подвыборками)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Скрипт для Google Colab:
- Загружает датасет ArabicNLPWorld/arabic-russian-parallel-corpus
- Создаёт JSON-датасет в формате:
  {
    "sources": ["предложение 1", "предложение 2", ...],
    "references": ["перевод 1", "перевод 2", ...],
    "source_lang": "ar",
    "target_lang": "ru"
  }
- Автоматически создаёт подвыборки: 200, 500, 1000, 10000 записей
- Сохраняет метаданные со статистикой
"""

import json
import random
from pathlib import Path

# Установка библиотеки datasets (если ещё не установлена)
!pip install datasets -q

from google.colab import userdata
from datasets import load_dataset

# ========== 1. ПОДКЛЮЧЕНИЕ ТОКЕНА ==========
print("=" * 60)
print("ШАГ 1: ПОДКЛЮЧЕНИЕ ТОКЕНА")
print("=" * 60)

try:
    FULL_HF_TOKEN = userdata.get('FULL_HF_TOKEN')
    print("✅ Токен успешно загружен из userdata")
except Exception as e:
    print(f"❌ Ошибка загрузки токена: {e}")
    FULL_HF_TOKEN = input("Введите ваш HF токен: ").strip()

# ========== 2. КОНФИГУРАЦИЯ ==========
print("\n" + "=" * 60)
print("ШАГ 2: КОНФИГУРАЦИЯ")
print("=" * 60)

DATASET_NAME = "ArabicNLPWorld/arabic-russian-parallel-corpus"
SPLIT = "train"  # или "full"
OUTPUT_DIR = Path("./arabic_russian_datasets")

# Размеры подвыборок
SAMPLE_SIZES = [200, 500, 1000, 10000]

# Языки
SOURCE_LANG = "ar"  # арабский
TARGET_LANG = "ru"  # русский

print(f"📌 Датасет: {DATASET_NAME}")
print(f"📌 Сплит: {SPLIT}")
print(f"📌 Выходная папка: {OUTPUT_DIR}")
print(f"📌 Язык источника: {SOURCE_LANG}")
print(f"📌 Язык перевода: {TARGET_LANG}")
print(f"📌 Размеры подвыборок: {SAMPLE_SIZES}")

# ========== 3. ЗАГРУЗКА ДАТАСЕТА ==========
print("\n" + "=" * 60)
print("ШАГ 3: ЗАГРУЗКА ДАТАСЕТА")
print("=" * 60)

print(f"📥 Загружаем датасет {DATASET_NAME} (split={SPLIT})...")
try:
    ds = load_dataset(DATASET_NAME, split=SPLIT, token=FULL_HF_TOKEN)
    print(f"✅ Датасет загружен! Количество записей: {len(ds):,}")
    print(f"📊 Колонки: {ds.column_names}")
except Exception as e:
    print(f"❌ Ошибка загрузки: {e}")
    raise

# ========== 4. ОПРЕДЕЛЕНИЕ КОЛОНОК ==========
print("\n" + "=" * 60)
print("ШАГ 4: ОПРЕДЕЛЕНИЕ КОЛОНОК")
print("=" * 60)

# Автоматическое определение названий колонок
if 'arabic' in ds.column_names and 'russian' in ds.column_names:
    SOURCE_COL = 'arabic'
    TARGET_COL = 'russian'
elif 'ar' in ds.column_names and 'ru' in ds.column_names:
    SOURCE_COL = 'ar'
    TARGET_COL = 'ru'
elif 'source' in ds.column_names and 'target' in ds.column_names:
    SOURCE_COL = 'source'
    TARGET_COL = 'target'
else:
    # Если колонки называются иначе, берём первые две
    SOURCE_COL = ds.column_names[0]
    TARGET_COL = ds.column_names[1]
    print(f"⚠️ Используем колонки: '{SOURCE_COL}' и '{TARGET_COL}'")

print(f"📌 Колонка источника: {SOURCE_COL}")
print(f"📌 Колонка перевода: {TARGET_COL}")

# Проверяем наличие колонки с источником (category)
HAS_SOURCE_COL = 'source' in ds.column_names

# ========== 5. ПРЕОБРАЗОВАНИЕ В СПИСКИ PYTHON ==========
print("\n" + "=" * 60)
print("ШАГ 5: ПРЕОБРАЗОВАНИЕ В СПИСКИ PYTHON")
print("=" * 60)

print("🔄 Преобразуем колонки в списки...")
source_texts = list(ds[SOURCE_COL])
target_texts = list(ds[TARGET_COL])

# Проверяем на пустые значения
empty_source = sum(1 for x in source_texts if x == "" or x is None)
empty_target = sum(1 for x in target_texts if x == "" or x is None)

print(f"✅ Преобразовано {len(source_texts):,} записей")
print(f"📊 Пустых записей в источнике: {empty_source}")
print(f"📊 Пустых записей в переводе: {empty_target}")

# ========== 6. СОЗДАНИЕ ПОЛНОГО JSON-ФАЙЛА ==========
print("\n" + "=" * 60)
print("ШАГ 6: СОЗДАНИЕ ПОЛНОГО JSON-ФАЙЛА")
print("=" * 60)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Создаём полный датасет
print(f"🔄 Создаём полный датасет ({SOURCE_LANG} → {TARGET_LANG})...")
full_dataset = {
    "sources": source_texts,
    "references": target_texts,
    "source_lang": SOURCE_LANG,
    "target_lang": TARGET_LANG
}
print(f"✅ Создано {len(full_dataset['sources']):,} записей")

# Сохраняем полный датасет
full_file = OUTPUT_DIR / f"{SOURCE_LANG}_to_{TARGET_LANG}_full.json"
print(f"💾 Сохраняем {full_file.name}...")
with open(full_file, "w", encoding="utf-8") as f:
    json.dump(full_dataset, f, ensure_ascii=False, indent=2)
print(f"✅ Файл сохранён: {full_file.name}")

# ========== 7. СОЗДАНИЕ ПОДВЫБОРОК ==========
print("\n" + "=" * 60)
print("ШАГ 7: СОЗДАНИЕ ПОДВЫБОРОК")
print("=" * 60)

def extract_sample(data, n, random_seed=42):
    """
    Извлекает случайную подвыборку из n записей

    Args:
        data: словарь с ключами 'sources', 'references', 'source_lang', 'target_lang'
        n: количество записей для извлечения
        random_seed: seed для воспроизводимости

    Returns:
        словарь с подвыборкой
    """
    total = len(data['sources'])

    if n >= total:
        print(f"⚠️ Запрошено {n} записей, но доступно только {total}. Берём все.")
        n = total

    # Создаём индексы и перемешиваем
    indices = list(range(total))
    random.seed(random_seed)
    random.shuffle(indices)
    selected_indices = indices[:n]

    # Создаём новую выборку
    sample = {
        "sources": [data['sources'][i] for i in selected_indices],
        "references": [data['references'][i] for i in selected_indices],
        "source_lang": data['source_lang'],
        "target_lang": data['target_lang']
    }

    return sample

# Создаём папку для подвыборок
samples_dir = OUTPUT_DIR / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)

print(f"📊 Создаём подвыборки в папке: {samples_dir}")
print("-" * 40)

sample_files = {}
for size in SAMPLE_SIZES:
    print(f"\n📊 Создаём подвыборку из {size:,} записей...")

    sample = extract_sample(full_dataset, size)
    sample_file = samples_dir / f"{SOURCE_LANG}_to_{TARGET_LANG}_{size}.json"

    with open(sample_file, "w", encoding="utf-8") as f:
        json.dump(sample, f, ensure_ascii=False, indent=2)

    print(f"   ✅ Создано {len(sample['sources']):,} записей -> {sample_file.name}")
    sample_files[str(size)] = {
        "file": sample_file.name,
        "num_examples": len(sample['sources'])
    }

# ========== 8. СОХРАНЕНИЕ МЕТАДАННЫХ ==========
print("\n" + "=" * 60)
print("ШАГ 8: СОХРАНЕНИЕ МЕТАДАННЫХ")
print("=" * 60)

metadata = {
    "full_dataset": {
        "source_lang": SOURCE_LANG,
        "target_lang": TARGET_LANG,
        "num_examples": len(full_dataset["sources"]),
        "file": full_file.name,
        "source_col": SOURCE_COL,
        "target_col": TARGET_COL,
        "empty_source": empty_source,
        "empty_target": empty_target
    },
    "samples": sample_files,
    "original_dataset": {
        "name": DATASET_NAME,
        "split": SPLIT,
        "total_records": len(ds),
        "column_names": ds.column_names
    }
}

# Добавляем статистику по источникам (если есть)
if HAS_SOURCE_COL:
    from collections import Counter
    sources = list(ds['source'])
    source_counts = Counter(sources)
    metadata["source_stats"] = {
        source: count for source, count in source_counts.most_common()
    }
    print("\n📊 СТАТИСТИКА ПО ИСТОЧНИКАМ:")
    for source, count in source_counts.most_common():
        percentage = (count / len(sources)) * 100
        print(f"   {source}: {count:,} ({percentage:.2f}%)")

# Добавляем статистику по длинам (если есть)
if 'arabic_char_count' in ds.column_names:
    arabic_chars = list(ds['arabic_char_count'])
    russian_chars = list(ds['russian_char_count'])
    metadata["char_stats"] = {
        "arabic": {
            "total": sum(arabic_chars),
            "average": sum(arabic_chars) / len(arabic_chars) if arabic_chars else 0,
            "min": min(arabic_chars) if arabic_chars else 0,
            "max": max(arabic_chars) if arabic_chars else 0
        },
        "russian": {
            "total": sum(russian_chars),
            "average": sum(russian_chars) / len(russian_chars) if russian_chars else 0,
            "min": min(russian_chars) if russian_chars else 0,
            "max": max(russian_chars) if russian_chars else 0
        }
    }

with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\n✅ Метаданные сохранены!")

# ========== 9. ПРОВЕРКА И СТАТИСТИКА ==========
print("\n" + "=" * 60)
print("ШАГ 9: СТАТИСТИКА И ПРОВЕРКА")
print("=" * 60)

print("\n📊 СТАТИСТИКА ДАТАСЕТА:")
print(f"   📁 Всего записей: {len(ds):,}")
print(f"   📁 Арабско-русский: {len(full_dataset['sources']):,} записей")
print(f"   📁 Пустых записей: {empty_source + empty_target}")

print("\n📁 СОЗДАННЫЕ ФАЙЛЫ:")
print(f"   📄 Полный датасет: {full_file.name} ({len(full_dataset['sources']):,} записей)")
print(f"   📁 Подвыборки в папке: {samples_dir}")
for size, info in sample_files.items():
    print(f"      📄 {info['file']} ({info['num_examples']:,} записей)")

# Проверяем первые 3 предложения
print("\n🔍 ПРИМЕРЫ (первые 3 предложения):")
print(f"\n--- {SOURCE_LANG.upper()} → {TARGET_LANG.upper()} ---")
for i in range(min(3, len(full_dataset["sources"]))):
    source_text = full_dataset['sources'][i][:100] + "..." if len(full_dataset['sources'][i]) > 100 else full_dataset['sources'][i]
    target_text = full_dataset['references'][i][:100] + "..." if len(full_dataset['references'][i]) > 100 else full_dataset['references'][i]
    print(f"  {i+1}. Источник ({SOURCE_LANG}): {source_text}")
    print(f"     Перевод ({TARGET_LANG}): {target_text}")
    print()

# Показываем структуру JSON
print("📄 СТРУКТУРА JSON:")
print(json.dumps({
    "sources": ["предложение 1", "предложение 2", "..."],
    "references": ["перевод 1", "перевод 2", "..."],
    "source_lang": SOURCE_LANG,
    "target_lang": TARGET_LANG
}, ensure_ascii=False, indent=2))

# ========== 10. СКАЧИВАНИЕ ФАЙЛОВ ==========
print("\n" + "=" * 60)
print("ШАГ 10: СКАЧИВАНИЕ ФАЙЛОВ")
print("=" * 60)

print(f"\n📁 Файлы сохранены в папке: {OUTPUT_DIR.absolute()}")

print("\n📥 Команды для скачивания:")
print("   from google.colab import files")
print(f"   files.download('{OUTPUT_DIR}/{full_file.name}')")
print("\n   # Подвыборки:")
for size, info in sample_files.items():
    print(f"   files.download('{OUTPUT_DIR}/samples/{info['file']}')")
print(f"   files.download('{OUTPUT_DIR}/metadata.json')")

# Автоматическое скачивание
download = input("\n⬇️ Скачать файлы автоматически? (y/n): ").strip().lower()
if download == 'y':
    from google.colab import files

    # Скачиваем полный датасет
    print(f"\n📥 Скачиваем {full_file.name}...")
    files.download(str(full_file))

    # Скачиваем подвыборки
    print("\n📥 Скачиваем подвыборки...")
    for size, info in sample_files.items():
        sample_file = samples_dir / info['file']
        if sample_file.exists():
            print(f"   Скачиваем {info['file']}...")
            files.download(str(sample_file))

    # Скачиваем метаданные
    print("\n📥 Скачиваем metadata.json...")
    files.download(str(OUTPUT_DIR / "metadata.json"))

    print("✅ Все файлы скачаны!")

print("\n" + "=" * 60)
print("🎉 ГОТОВО!")
print("=" * 60)

# ========== 11. ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ ==========
print("\n" + "=" * 60)
print("📊 ИТОГОВАЯ ИНФОРМАЦИЯ О ДАТАСЕТЕ")
print("=" * 60)

print(f"""
📋 Сводка:
   • Всего записей: {len(ds):,}
   • Язык источника: {SOURCE_LANG} ({SOURCE_COL})
   • Язык перевода: {TARGET_LANG} ({TARGET_COL})
   • Источников данных: {len(source_counts) if HAS_SOURCE_COL else 'не указано'}
""")

if HAS_SOURCE_COL:
    print("📂 Распределение по источникам:")
    for source, count in source_counts.most_common():
        percentage = (count / len(sources)) * 100
        bar = "█" * int(percentage / 2)
        print(f"   {source:15} {bar} {count:,} ({percentage:.1f}%)")

print(f"""
📁 Сохранённые файлы:
   • Полный датасет: {full_file.name} ({len(full_dataset['sources']):,} записей)
   • Подвыборки: {len(sample_files)} файлов
   • Метаданные: metadata.json
""")